# Ch.4 — Parallelism & Distributed Training (Azure ML Supplement)

**No local GPU required.** This notebook shows how to run distributed training (PyTorch DDP + LoRA) on Azure ML compute clusters. You'll submit jobs to multi-node GPU clusters and monitor training remotely.

**What this notebook demonstrates:**
1. Azure ML workspace connection and authentication
2. Multi-node GPU cluster setup (2-8 nodes with A100 or V100 GPUs)
3. PyTorch DistributedDataParallel (DDP) configuration for Azure ML
4. LoRA fine-tuning job submission to Azure ML compute
5. Training job monitoring and cost tracking
6. Model artifact download from Azure ML
7. Cost comparison: Azure ML vs local RTX 4090

**Prerequisites:**
- Azure subscription with Azure ML workspace created
- Azure ML SDK v2 installed: `pip install azure-ai-ml azure-identity`
- (Optional) Azure CLI: `az login` for authentication

**WARNING:** Multi-node GPU training on Azure ML incurs costs ($2-10/hour per node depending on GPU type). Monitor your spending carefully.

## 1 · Azure ML Credentials Setup

**IMPORTANT SECURITY NOTE:** The pre-push Git hook will strip these credentials before any commit reaches the remote repository. However, you should still:
- Never commit credentials to personal repositories
- Use Azure Key Vault for production workloads
- Rotate keys regularly
- Use Managed Identity when running on Azure compute

**To get these values:**
1. Go to Azure Portal → Machine Learning → Your Workspace
2. Click "Overview" to find: Subscription ID, Resource Group, Workspace Name
3. For region: Check "Location" field (e.g., `eastus`, `westus2`, `westeurope`)

In [ ]:
# ── Azure ML Credentials ──────────────────────────────────────────────────
# USER: Replace with your Azure ML workspace details
# These will be stripped by pre-push hook before commit

AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID (GUID)
AZURE_RESOURCE_GROUP = ""   # Resource group containing your ML workspace
AZURE_WORKSPACE_NAME = ""   # Your Azure ML workspace name
AZURE_REGION = "eastus"     # Azure region (e.g., eastus, westus2, westeurope)

# Validation
if not all([AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, AZURE_WORKSPACE_NAME]):
    print("⚠️  WARNING: Azure credentials not set. This notebook will fail.")
    print("   Fill in AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, and AZURE_WORKSPACE_NAME above.")
else:
    print("✅ Azure credentials configured.")
    print(f"   Subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"   Resource Group: {AZURE_RESOURCE_GROUP}")
    print(f"   Workspace: {AZURE_WORKSPACE_NAME}")
    print(f"   Region: {AZURE_REGION}")

## 2 · Azure ML SDK Setup

Install dependencies and import Azure ML SDK v2.

**Installation (run once in your environment):**
```bash
pip install azure-ai-ml azure-identity
```

In [ ]:
# ── Azure ML SDK Imports ──────────────────────────────────────────────────
try:
    from azure.ai.ml import MLClient
    from azure.ai.ml.entities import (
        AmlCompute,
        Command,
        Environment,
        MpiDistribution,
        PyTorchDistribution,
    )
    from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
    import json
    import time
    print("✅ Azure ML SDK v2 loaded successfully.")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Run: pip install azure-ai-ml azure-identity")
    raise

## 3 · Connect to Azure ML Workspace

Authenticate and connect to your Azure ML workspace.

**Authentication methods:**
1. **DefaultAzureCredential** (recommended): Tries multiple auth methods (CLI, Managed Identity, etc.)
2. **InteractiveBrowserCredential**: Opens browser for manual login

If you've run `az login` in your terminal, `DefaultAzureCredential` should work automatically.

In [ ]:
# ── Connect to Azure ML Workspace ────────────────────────────────────────
try:
    # Try DefaultAzureCredential first (works with az login)
    credential = DefaultAzureCredential()
    
    ml_client = MLClient(
        credential=credential,
        subscription_id=AZURE_SUBSCRIPTION_ID,
        resource_group_name=AZURE_RESOURCE_GROUP,
        workspace_name=AZURE_WORKSPACE_NAME,
    )
    
    # Test connection by fetching workspace details
    workspace = ml_client.workspaces.get(name=AZURE_WORKSPACE_NAME)
    
    print("✅ Connected to Azure ML workspace successfully!")
    print(f"   Workspace: {workspace.name}")
    print(f"   Location: {workspace.location}")
    print(f"   ID: {workspace.id[:60]}...")
    
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Run 'az login' in terminal to authenticate with Azure CLI")
    print("2. Verify your subscription ID, resource group, and workspace name are correct")
    print("3. Check that you have Contributor access to the Azure ML workspace")
    print("\nAlternatively, try InteractiveBrowserCredential for manual login.")
    raise

## 4 · Create Multi-Node GPU Compute Cluster

Azure ML compute clusters can scale from 0 to N nodes automatically. We'll create a cluster configured for distributed training.

**Cluster configuration:**
- **VM size:** `Standard_NC24ads_A100_v4` (1× A100 80GB per node) or `Standard_NC6s_v3` (1× V100 16GB, cheaper)
- **Min nodes:** 0 (scales down to zero when idle → no cost)
- **Max nodes:** 4 (can run 4-node distributed training)
- **Idle time:** 120 seconds (how long to wait before scaling down)

**Cost estimates (as of 2025):**
- `Standard_NC24ads_A100_v4` (A100 80GB): ~$3.67/hour per node
- `Standard_NC6s_v3` (V100 16GB): ~$0.90/hour per node
- `Standard_NC12s_v3` (2× V100 16GB): ~$1.80/hour per node

In [ ]:
# ── Create Azure ML Compute Cluster ──────────────────────────────────────
cluster_name = "gpu-cluster-ddp"

# Check if cluster already exists
try:
    cluster = ml_client.compute.get(cluster_name)
    print(f"✅ Compute cluster '{cluster_name}' already exists.")
    print(f"   VM Size: {cluster.size}")
    print(f"   Min nodes: {cluster.min_instances}")
    print(f"   Max nodes: {cluster.max_instances}")
    print(f"   Current state: {cluster.provisioning_state}")
except Exception:
    print(f"⏳ Creating compute cluster '{cluster_name}'...")
    
    cluster = AmlCompute(
        name=cluster_name,
        type="amlcompute",
        size="Standard_NC6s_v3",  # 1× V100 16GB (change to Standard_NC24ads_A100_v4 for A100)
        min_instances=0,  # Scale to zero when idle (no cost)
        max_instances=4,  # Maximum 4 nodes for distributed training
        idle_time_before_scale_down=120,  # Wait 2 minutes before scaling down
        tier="dedicated",  # Use dedicated VMs (not low-priority)
    )
    
    cluster = ml_client.compute.begin_create_or_update(cluster).result()
    
    print("✅ Compute cluster created successfully!")
    print(f"   Name: {cluster.name}")
    print(f"   VM Size: {cluster.size}")
    print(f"   Max nodes: {cluster.max_instances}")
    print(f"\n💡 Cluster will auto-scale from 0 to {cluster.max_instances} nodes as needed.")
    print(f"   Estimated cost: ~$0.90/hour per V100 node (only when running jobs)")

## 5 · PyTorch DDP Training Script (Azure ML Compatible)

This cell creates the training script that will run on each node in the distributed cluster.

**Key differences from local DDP:**
1. Azure ML sets environment variables: `RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`
2. Use `PyTorchDistribution` in job config to handle process initialization
3. Each node runs multiple processes (one per GPU)

**LoRA configuration:**
- Rank: 16 (sweet spot for quality vs memory)
- Target modules: attention layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`)
- Trainable params: ~0.5% of Llama-3-8B (42M / 8B)

In [ ]:
# ── Create Training Script ───────────────────────────────────────────────
training_script = """
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import argparse

def setup_distributed():
    \"\"\"Initialize distributed training environment (Azure ML provides env vars).\"\"\"  
    # Azure ML sets these environment variables automatically
    rank = int(os.environ.get('RANK', 0))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    
    # Initialize process group (NCCL backend for GPU training)
    dist.init_process_group(
        backend='nccl',
        init_method='env://',  # Use environment variables
        world_size=world_size,
        rank=rank,
    )
    
    # Set device for this process
    torch.cuda.set_device(local_rank)
    
    if rank == 0:
        print(f"✅ Distributed training initialized:")
        print(f"   World size (total GPUs): {world_size}")
        print(f"   Current rank: {rank}")
        print(f"   Local rank (GPU on this node): {local_rank}")
    
    return rank, world_size, local_rank

def main(args):
    # Setup distributed environment
    rank, world_size, local_rank = setup_distributed()
    
    # Load base model (only rank 0 prints to avoid clutter)
    if rank == 0:
        print("\n⏳ Loading Llama-3-8B base model...")
    
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Meta-Llama-3-8B-Instruct",
        torch_dtype=torch.float16,
        device_map=None,  # We'll handle device placement manually
    )
    model = model.to(f'cuda:{local_rank}')
    
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
    tokenizer.pad_token = tokenizer.eos_token
    
    # Apply LoRA adapters
    if rank == 0:
        print("\n⏳ Applying LoRA adapters (rank=16)...")
    
    lora_config = LoraConfig(
        r=16,  # Rank (controls adapter size)
        lora_alpha=32,  # Scaling factor
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    
    model = get_peft_model(model, lora_config)
    
    if rank == 0:
        model.print_trainable_parameters()
        print(f"   Expected: ~0.5% trainable (42M / 8B params)")
    
    # Load dataset (synthetic example - replace with your data)
    if rank == 0:
        print("\n⏳ Loading training dataset...")
    
    # For demo: use a small dataset. In production, replace with your invoice dataset.
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    
    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    
    # Training arguments (configured for distributed training)
    training_args = TrainingArguments(
        output_dir="./llama3-lora-ddp",
        per_device_train_batch_size=4,  # Batch size per GPU
        gradient_accumulation_steps=4,  # Effective batch = 4 × 4 × world_size
        num_train_epochs=1,  # Short for demo (use 3+ in production)
        learning_rate=2e-4,
        fp16=True,  # Use mixed precision
        logging_steps=10,
        save_steps=100,
        warmup_steps=50,
        save_total_limit=2,
        report_to="none",  # Disable wandb/tensorboard for simplicity
        local_rank=local_rank,  # Required for DDP
    )
    
    # Create trainer (Hugging Face Trainer handles DDP automatically)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
    )
    
    # Train!
    if rank == 0:
        print(f"\n🚀 Starting distributed training on {world_size} GPUs...")
        print(f"   Per-GPU batch: {training_args.per_device_train_batch_size}")
        print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
        print(f"   Effective batch: {4 * 4 * world_size} (per_batch × accum × world_size)")
    
    trainer.train()
    
    # Save model (only rank 0 saves to avoid conflicts)
    if rank == 0:
        print("\n✅ Training complete! Saving LoRA adapters...")
        model.save_pretrained("./outputs/lora-adapters")
        tokenizer.save_pretrained("./outputs/lora-adapters")
        print("   Adapters saved to: ./outputs/lora-adapters")
    
    # Cleanup
    dist.destroy_process_group()

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_path", type=str, default="", help="Path to training data")
    args = parser.parse_args()
    
    main(args)
"""

# Write training script to file
with open("train_lora_ddp.py", "w") as f:
    f.write(training_script)

print("✅ Training script created: train_lora_ddp.py")
print("   This script will run on each node in the Azure ML compute cluster.")
print("   Key features:")
print("   - PyTorch DistributedDataParallel (DDP) for multi-GPU training")
print("   - LoRA adapters (rank=16) for memory-efficient fine-tuning")
print("   - Gradient accumulation (effective batch=64 with 4 GPUs)")
print("   - Mixed precision (FP16) to reduce memory usage")

## 6 · Submit Distributed Training Job to Azure ML

Now we submit the training job to our GPU cluster. Azure ML will:
1. Spin up the requested number of nodes (scales from 0 → 2 nodes)
2. Install dependencies on each node
3. Run the training script with PyTorch DDP configuration
4. Collect logs and outputs
5. Scale down to 0 nodes after job completes (no idle cost)

**Job configuration:**
- **Instance count:** 2 nodes (each with 1 V100 GPU) → 2 GPUs total
- **Process count per node:** 1 (since Standard_NC6s_v3 has 1 GPU)
- **Distribution:** PyTorchDistribution (handles DDP process spawning)
- **Environment:** PyTorch 2.0 + Transformers + PEFT

In [ ]:
# ── Submit Distributed Training Job ──────────────────────────────────────
job_name = "llama3-lora-ddp-training"

# Define custom environment with required packages
environment = Environment(
    name="pytorch-lora-ddp",
    description="PyTorch 2.0 + Transformers + PEFT for LoRA distributed training",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04",  # Base image with CUDA
    conda_file="environment.yml",  # We'll create this below
)

# Create conda environment file
conda_env = """
name: pytorch-lora
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pip
  - pip:
    - torch>=2.0.0
    - transformers>=4.36.0
    - peft>=0.7.0
    - datasets>=2.14.0
    - accelerate>=0.25.0
    - bitsandbytes>=0.41.0
"""

with open("environment.yml", "w") as f:
    f.write(conda_env)

print("✅ Environment configuration created.")

# Create distributed training job
job = Command(
    display_name="Llama-3-8B LoRA Fine-Tuning (DDP)",
    description="Distributed LoRA fine-tuning on 2 nodes with PyTorch DDP",
    code="./",  # Upload current directory (contains train_lora_ddp.py)
    command="python train_lora_ddp.py",
    environment=environment,
    compute=cluster_name,
    instance_count=2,  # 2 nodes × 1 GPU/node = 2 GPUs total
    distribution=PyTorchDistribution(
        process_count_per_instance=1,  # 1 GPU per node (Standard_NC6s_v3)
    ),
    experiment_name="llama3-lora-distributed",
)

print("\n⏳ Submitting job to Azure ML...")
print(f"   Compute cluster: {cluster_name}")
print(f"   Instance count: 2 nodes (2 GPUs total)")
print(f"   Estimated cost: ~$1.80/hour (2 × V100 nodes)")
print(f"   Expected duration: 30-60 minutes (demo dataset)")

# Submit job
submitted_job = ml_client.jobs.create_or_update(job)

print(f"\n✅ Job submitted successfully!")
print(f"   Job name: {submitted_job.name}")
print(f"   Status: {submitted_job.status}")
print(f"   Studio URL: {submitted_job.studio_url}")
print(f"\n💡 Click the URL above to monitor training in Azure ML Studio.")
print(f"   The cluster will auto-scale from 0 → 2 nodes → 0 after completion.")

## 7 · Monitor Training Job

Poll the job status and display live updates. You can also monitor in Azure ML Studio (click the URL from previous cell).

**Job lifecycle:**
1. **Queued:** Waiting for compute nodes to spin up
2. **Preparing:** Installing environment (conda packages)
3. **Running:** Training in progress
4. **Completed:** Success! Artifacts available for download
5. **Failed:** Check logs in Azure ML Studio for error details

**Typical timeline:**
- Node provisioning: 3-5 minutes
- Environment setup: 5-10 minutes (first run; cached on subsequent runs)
- Training: 30-60 minutes (depends on dataset size and epochs)

In [ ]:
# ── Monitor Training Job ─────────────────────────────────────────────────
print(f"⏳ Monitoring job: {submitted_job.name}\n")

start_time = time.time()
last_status = None

while True:
    # Refresh job status
    job_status = ml_client.jobs.get(submitted_job.name)
    
    # Print status update if changed
    if job_status.status != last_status:
        elapsed = time.time() - start_time
        print(f"[{elapsed/60:.1f} min] Status: {job_status.status}")
        
        if job_status.status == "Preparing":
            print("   → Nodes are spinning up and installing dependencies...")
        elif job_status.status == "Running":
            print("   → Training started! Check Azure ML Studio for live logs.")
            print(f"   → Studio URL: {job_status.studio_url}")
        
        last_status = job_status.status
    
    # Check if job finished
    if job_status.status in ["Completed", "Failed", "Canceled"]:
        break
    
    # Wait 30 seconds before next poll
    time.sleep(30)

# Final status
elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"Job finished: {job_status.status}")
print(f"Total time: {elapsed/60:.1f} minutes")

if job_status.status == "Completed":
    print("\n✅ Training completed successfully!")
    print("   Outputs saved in: ./outputs/lora-adapters")
    print("   Next: Download model artifacts (see next cell)")
else:
    print(f"\n❌ Job {job_status.status.lower()}.")
    print("   Check logs in Azure ML Studio for error details.")
    print(f"   Studio URL: {job_status.studio_url}")

## 8 · Download Trained LoRA Adapters

After training completes, download the LoRA adapter weights from Azure ML.

**What you get:**
- `adapter_model.bin`: LoRA adapter weights (~84 MB for rank=16)
- `adapter_config.json`: LoRA configuration (rank, target modules, etc.)
- `tokenizer_config.json`, `tokenizer.json`: Tokenizer files

**Local inference:**
```python
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, "./downloaded-adapters")

# Run inference
outputs = model.generate(...)
```

In [ ]:
# ── Download Model Artifacts ─────────────────────────────────────────────
if job_status.status == "Completed":
    print("⏳ Downloading trained LoRA adapters from Azure ML...\n")
    
    # Download outputs folder
    ml_client.jobs.download(
        name=submitted_job.name,
        download_path="./azure-ml-outputs",
        output_name="outputs",  # Default output folder
    )
    
    print("✅ Adapters downloaded successfully!")
    print("   Location: ./azure-ml-outputs/outputs/lora-adapters")
    print("\n📦 Downloaded files:")
    
    import os
    adapter_dir = "./azure-ml-outputs/outputs/lora-adapters"
    if os.path.exists(adapter_dir):
        for file in os.listdir(adapter_dir):
            file_path = os.path.join(adapter_dir, file)
            size_mb = os.path.getsize(file_path) / (1024**2)
            print(f"   - {file} ({size_mb:.1f} MB)")
    
    print("\n💡 To use these adapters locally:")
    print("   1. Load base model: AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct')")
    print("   2. Load adapters: PeftModel.from_pretrained(base_model, './azure-ml-outputs/outputs/lora-adapters')")
    print("   3. Run inference: model.generate(...)")
else:
    print("⚠️  Job did not complete successfully. Cannot download artifacts.")
    print(f"   Job status: {job_status.status}")

## 9 · Cost Comparison: Azure ML vs Local RTX 4090

Let's compare the economics of distributed training on Azure ML vs purchasing local hardware.

**Scenario:** Fine-tune Llama-3-8B with LoRA for 2 days

### Azure ML (2× Standard_NC6s_v3, V100 16GB)
- Hourly cost: $0.90/hour × 2 nodes = $1.80/hour
- Training time: 2 days (48 hours)
- **Total cost: $86.40** (one-time, no upfront investment)
- Pros: Pay-per-use, no maintenance, instant scaling, latest GPUs
- Cons: Slower than A100, data transfer costs, egress fees

### Azure ML (2× Standard_NC24ads_A100_v4, A100 80GB)
- Hourly cost: $3.67/hour × 2 nodes = $7.34/hour
- Training time: 1 day (24 hours, 2× faster than V100)
- **Total cost: $176.16** (one-time)
- Pros: Fastest training, 80GB VRAM (can do full fine-tuning), ZeRO-3 capable
- Cons: Higher hourly cost

### Local Hardware (2× RTX 4090, 24GB)
- Upfront cost: $1,600 × 2 = $3,200 (GPUs only; add $1,000+ for mobo/CPU/PSU/cooling)
- Training time: 2 days (48 hours)
- Electricity: ~900W × 48h × $0.12/kWh = $5.18
- **Total cost: $4,200 upfront + $5.18 per training run**
- Break-even: $4,200 / $86.40 = **49 training runs** (vs V100 on Azure)
- Pros: Unlimited runs after break-even, no data transfer, local control
- Cons: Upfront investment, maintenance, depreciation, single location

### Recommendation
- **Experimentation phase (1-10 runs):** Use Azure ML V100 ($86/run) — no upfront cost, fast iteration
- **Production phase (50+ runs/year):** Purchase RTX 4090s ($4,200 upfront) — break-even at 49 runs
- **Large models (70B+):** Use Azure ML A100 80GB ($176/run) — cannot fit on RTX 4090
- **Burst workloads:** Combine local RTX 4090 (baseline) + Azure ML (overflow) for cost optimization

In [ ]:
# ── Cost Calculator ──────────────────────────────────────────────────────
import pandas as pd

# Cost parameters
training_hours = 48  # 2 days for demo dataset (real invoice training may be longer)
electricity_kwh_cost = 0.12  # USD per kWh (adjust for your region)

cost_data = [
    # (Platform, Setup, GPU Count, GPU Type, $/hour, Training Hours, Upfront Cost)
    ("Azure ML", "2× V100 16GB", 2, "V100", 0.90 * 2, training_hours, 0),
    ("Azure ML", "2× A100 80GB", 2, "A100", 3.67 * 2, training_hours / 2, 0),  # 2× faster
    ("Azure ML", "4× V100 16GB", 4, "V100", 0.90 * 4, training_hours / 2, 0),  # 2× faster with 4 GPUs
    ("Local", "2× RTX 4090 24GB", 2, "RTX 4090", (900 / 1000) * electricity_kwh_cost, training_hours, 4200),
    ("Local", "4× RTX 4090 24GB", 4, "RTX 4090", (1800 / 1000) * electricity_kwh_cost, training_hours / 2, 8400),
]

results = []
for platform, setup, gpu_count, gpu_type, hourly_cost, hours, upfront in cost_data:
    training_cost = hourly_cost * hours
    total_first_run = upfront + training_cost
    results.append({
        "Platform": platform,
        "Setup": setup,
        "GPU Type": gpu_type,
        "Training Hours": hours,
        "Upfront Cost": f"${upfront:,.0f}",
        "Training Cost": f"${training_cost:.2f}",
        "Total (1st run)": f"${total_first_run:,.2f}",
    })

cost_df = pd.DataFrame(results)
print("\n💰 Cost Comparison: Azure ML vs Local Hardware\n")
print(cost_df.to_string(index=False))

print("\n📊 Break-Even Analysis:")
azure_v100_cost = 0.90 * 2 * training_hours
local_4090_upfront = 4200
local_4090_per_run = (900 / 1000) * electricity_kwh_cost * training_hours

break_even_runs = local_4090_upfront / (azure_v100_cost - local_4090_per_run)
print(f"   Break-even point: {break_even_runs:.0f} training runs")
print(f"   (After {break_even_runs:.0f} runs, local RTX 4090 becomes cheaper than Azure V100)")

print("\n💡 Decision Guide:")
print(f"   • 1-49 runs:    Use Azure ML V100 (${azure_v100_cost:.2f}/run, no upfront cost)")
print(f"   • 50+ runs:     Purchase local RTX 4090 (${local_4090_upfront:,} upfront + ${local_4090_per_run:.2f}/run)")
print(f"   • 70B+ models:  Use Azure ML A100 80GB (${3.67 * 2 * (training_hours/2):.2f}/run, 80GB VRAM required)")

## 10 · Advanced: ZeRO-2 Distributed Training (4+ Nodes)

For larger models or full fine-tuning (not just LoRA), you can use **DeepSpeed ZeRO-2** to shard optimizer states across GPUs.

**When to use ZeRO-2:**
- Full fine-tuning (not LoRA): Need to update all 8B parameters
- Memory-constrained: 24GB VRAM not enough for full fine-tuning (104GB required)
- 4+ GPUs available: ZeRO-2 shards optimizer across GPUs (64GB ÷ 4 = 16GB per GPU)

**Memory breakdown with ZeRO-2 (4× V100 16GB):**
- Parameters (FP16, replicated): 16 GB per GPU
- Optimizer states (sharded): 16 GB per GPU (64GB total ÷ 4)
- Gradients (sharded): 4 GB per GPU (16GB total ÷ 4)
- Activations: 8 GB per GPU
- **Total: 44 GB per GPU** ❌ (still exceeds 16GB V100)

**Solution:** Use A100 40GB or enable gradient checkpointing (reduces activations 75%).

This cell shows how to configure DeepSpeed ZeRO-2 for Azure ML.

In [ ]:
# ── DeepSpeed ZeRO-2 Configuration ───────────────────────────────────────
deepspeed_config = {
    "train_batch_size": "auto",  # Auto-calculated from per_device_batch × world_size × grad_accum
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": 1.0,
    "fp16": {
        "enabled": True,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "min_loss_scale": 1,
    },
    "zero_optimization": {
        "stage": 2,  # ZeRO-2: Shard optimizer states + gradients
        "offload_optimizer": {
            "device": "none",  # Keep optimizer on GPU (use "cpu" if OOM)
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
    },
    "scheduler": {
        "type": "WarmupDecayLR",
        "params": {
            "warmup_min_lr": 0,
            "warmup_max_lr": 2e-5,
            "warmup_num_steps": 100,
            "total_num_steps": 1000,
        },
    },
}

# Save DeepSpeed config
import json
with open("deepspeed_config.json", "w") as f:
    json.dump(deepspeed_config, f, indent=2)

print("✅ DeepSpeed ZeRO-2 configuration created: deepspeed_config.json")
print("\n📝 Key settings:")
print("   - ZeRO stage: 2 (shard optimizer + gradients)")
print("   - Optimizer offload: none (keep on GPU for speed)")
print("   - FP16: enabled (mixed precision training)")
print("   - Gradient clipping: 1.0 (prevent exploding gradients)")
print("\n💡 To use ZeRO-2 in your training script:")
print("   1. Add: from transformers import TrainingArguments")
print("   2. Set: training_args.deepspeed = 'deepspeed_config.json'")
print("   3. Submit job with instance_count=4 (4 nodes for ZeRO-2 sharding)")
print("\n⚠️  Memory requirements with ZeRO-2 (4 nodes):")
print("   - Per GPU: 44 GB (requires A100 40GB or 80GB)")
print("   - Cannot fit on V100 16GB or RTX 4090 24GB without gradient checkpointing")

## 11 · Cluster Cleanup

After completing your experiments, you can delete the compute cluster to ensure no accidental costs.

**Note:** Our cluster is configured with `min_instances=0`, so it scales down automatically when idle. You only pay when jobs are running. Deleting the cluster is optional — it's safe to leave it for future use.

**When to delete:**
- Finished with experiments for the quarter
- Need to free up subscription quota for other resources
- Want to recreate cluster with different VM size

**To delete:** Uncomment and run the cell below.

In [ ]:
# ── Delete Compute Cluster (Optional) ────────────────────────────────────
# Uncomment to delete the cluster

# print(f"⏳ Deleting compute cluster '{cluster_name}'...")
# ml_client.compute.begin_delete(cluster_name).result()
# print(f"✅ Cluster '{cluster_name}' deleted successfully.")
# print("   No compute costs will be incurred.")

print("💡 Cluster deletion is commented out by default.")
print("   Your cluster scales to zero when idle (no cost).")
print("   Uncomment the lines above to permanently delete the cluster.")

## Summary & Next Steps

**What you learned:**
1. ✅ Azure ML workspace connection and authentication
2. ✅ Multi-node GPU cluster creation (auto-scaling 0 → N nodes)
3. ✅ PyTorch DistributedDataParallel (DDP) training on Azure ML
4. ✅ LoRA fine-tuning for memory-efficient distributed training
5. ✅ Job submission, monitoring, and artifact download
6. ✅ Cost comparison: Azure ML vs local hardware
7. ✅ DeepSpeed ZeRO-2 configuration for large-scale training

**Key takeaways:**
- **LoRA + DDP:** Train Llama-3-8B on 2× V100 16GB for $86 (2 days)
- **Break-even:** 49 training runs (Azure V100 vs local RTX 4090)
- **Scaling:** Azure ML auto-scales from 0 → N nodes → 0 (pay per use)
- **ZeRO-2:** Full fine-tuning requires A100 40GB+ (44GB per GPU with 4-node sharding)

**Next steps:**
1. **Ch.5 (Inference Optimization):** Deploy your fine-tuned LoRA adapters with vLLM
2. **Production:** Set up Azure ML pipelines for automated retraining
3. **Monitoring:** Integrate with Azure Monitor for cost tracking and alerting
4. **Experimentation:** Try ZeRO-3 (shard parameters) for 70B+ models

**Resources:**
- [Azure ML documentation](https://docs.microsoft.com/en-us/azure/machine-learning/)
- [PyTorch DDP tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- [PEFT (LoRA) library](https://github.com/huggingface/peft)
- [DeepSpeed ZeRO](https://www.deepspeed.ai/tutorials/zero/)